[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-knn.ipynb)

# K-Nearest Neighbours

*AIBits Academy · Machine Learning End To End · Instance-Based Learning*

A non-parametric, lazy learning algorithm that classifies (or regresses) based on the k most similar training samples — no explicit training phase required.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

> **🎯 Intuition First**
>
> Ask a new arrival in Bengaluru's tech corridor to guess a stranger's job, and they'll instinctively look at who that stranger hangs around with most — not run a formula. KNN formalises exactly that instinct: a new point is classified as whatever its **k nearest neighbours** mostly are. There's no equation to fit and no training phase — the entire "model" is just the labelled examples themselves, sitting in memory. All the real work happens at prediction time, when you go looking for the closest matches to a brand-new point.

> **📋 Real-World Case Study — RFM Customer Segmentation**
>
> A common retail/e-commerce use case: score every customer on **Recency** (days since last purchase), **Frequency** (number of purchases), and **Monetary value** (total spend) — three numeric features per customer. KNN then groups a new or ambiguous customer with their nearest neighbours in this 3D RFM space, giving marketing teams an intuitive, explainable way to segment customers ("this customer behaves like our high-value repeat-buyer cluster") without needing to hand-craft segmentation rules.

## How KNN Works

KNN is called a **lazy learner** because it memorises the training set instead of learning a model. At prediction time for a new point x:

1. Compute distance from x to every training point (Euclidean, Manhattan, Minkowski…)
2. Select the k nearest neighbours
3. **Classification:** majority vote among k neighbours → predicted class
4. **Regression:** average (or weighted average) of k neighbours' y values → ŷ

## Distance Metrics

$$\begin{gathered}\text{Euclidean:}\quad d(a,b) = \sqrt{\sum (a_i-b_i)^2}\\[6pt]\text{Manhattan:}\quad d(a,b) = \sum |a_i-b_i|\\[6pt]\text{Minkowski:}\quad d(a,b) = \Big(\sum |a_i-b_i|^p\Big)^{1/p}\end{gathered}$$

## Interactive KNN Visualisation

Classify new Bengaluru startup employees (experience in years, monthly CTC in ₹ lakhs) as Junior / Senior. Try different k values:

## From Scratch with NumPy

In [ ]:
# KNN from scratch — Flipkart customer category prediction
import numpy as np
from collections import Counter

class KNNClassifier:
    def __init__(self, k=3, metric='euclidean'):
        self.k = k; self.metric = metric

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def _dist(self, a, b):
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((a-b)**2, axis=1))
        return np.sum(np.abs(a-b), axis=1)   # manhattan

    def predict(self, X):
        X = np.array(X)
        preds = []
        for x in X:
            dists = self._dist(self.X_train, x)
            idx   = np.argsort(dists)[:self.k]
            vote  = Counter(self.y_train[idx]).most_common(1)[0][0]
            preds.append(vote)
        return np.array(preds)

# Flipkart user data: [avg_order_val, sessions/week, cart_abandon_rate] → segment
np.random.seed(0)
X = np.random.rand(200, 3)
y = (X[:,0] + X[:,1] > 1.0).astype(int)  # 1=High-value, 0=Casual

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.25,random_state=42)
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)

for k in [1,3,5,7,11]:
    m = KNNClassifier(k=k)
    m.fit(X_tr_s, y_tr)
    acc = np.mean(m.predict(X_te_s)==y_te)
    print(f"k={k:2d}  Accuracy={acc:.3f}")

## Choosing k — the Elbow Method

Plot error rate vs k on validation set. The optimal k is where the error rate stops decreasing rapidly (elbow). Always try odd k for binary classification to avoid ties. A rule of thumb is k ≈ √n (where n = number of training samples), then fine-tune with cross-validation.

## k=1 vs Large k — Seeing the Bias-Variance Trade-off

The interactive plot above lets you drag the k slider and watch the shaded decision region update live. Placing the two extremes side by side makes the trade-off concrete. This uses a slightly noisier version of the same Junior/Senior dataset — including two deliberately mislabelled/outlier points — specifically so the effect is visible:

With **k=1**, each outlier point carves out its own tiny island of the "wrong" colour — the model has essentially zero bias (it fits every training point exactly) but high variance (one noisy point flips an entire neighbourhood's prediction). With **k=11**, votes are averaged over enough neighbours that the two outliers get outvoted and smoothed away entirely — lower variance, at the cost of a little bias right at the genuine class boundary. It's the same underfit/overfit trade-off seen on the Polynomial Regression and Regularization pages, just expressed through a neighbour-count hyperparameter instead of a model-complexity term.

## Time & Space Complexity

| Operation | Brute-force KNN | KD-Tree / Ball Tree |
|---|---|---|
| Training | O(1) | O(n log n) |
| Prediction (1 point) | O(n·p) | O(p log n) average |
| Storage | O(n·p) | O(n·p) |

## When to Use / Avoid

### ✓ Use KNN when

- Dataset is small to medium (< 50k samples)
- No assumptions about data distribution
- Non-linear decision boundaries needed
- Interpretability of "similar cases" is valuable
- Recommendation systems (similar users/items)

### ✗ Avoid when

- Large datasets (O(n) prediction is slow)
- High-dimensional features (curse of dimensionality)
- Features are not scaled (distance is distorted)
- Many irrelevant features (noise dominates distance)
- Real-time inference required at scale

## The Curse of Dimensionality — Why "Nearest" Stops Meaning Local

KNN's core assumption is that points close together in feature space are similar in label. That assumption quietly breaks down as the number of features grows. Picture carving out a sub-region of feature space meant to capture some fixed fraction p of all the data — say 10%. In 1 dimension that's just a slice covering 10% of the range. In higher dimensions, capturing that same 10% of the data forces the sub-region's edge length to stretch out much further, because volume grows so much faster than length does. The chart below plots the edge length needed to capture 10% and 50% of the data as dimensionality increases:

By 15 dimensions, capturing just 10% of the data needs a sub-region spanning 86% of each feature's range — and capturing half the data needs 95% of the range. There's no "local" neighbourhood left; every point is nearly as far away as every other point. This is precisely why the Conceptual Q&A note below about points "becoming equidistant" happens in practice, and why dimensionality reduction (PCA) or careful feature selection often becomes mandatory before applying KNN to wide datasets — for example, a Zomato dish-recommendation feature set with hundreds of embedding dimensions.

> **🔗 Real-World Link — News Recommendation**
>
> 180 news articles, each already reduced to a numeric embedding vector — recommending similar articles becomes a plain KNN nearest-neighbour search over those vectors, no text processing needed. [See the case study →](https://statso.io/news-recommendation-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Find the nearest neighbour

Compute the Euclidean distance from `query` to every row of `X` and store the **index of the closest row** in `nearest`.

In [ ]:
import numpy as np
X = np.array([[1.0, 2.0], [5.0, 6.0], [2.0, 2.5], [9.0, 9.0]])
query = np.array([2.2, 2.4])
nearest = None   # TODO


In [ ]:
try:
    check("closest row is #2", nearest == 2)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
X = np.array([[1.0, 2.0], [5.0, 6.0], [2.0, 2.5], [9.0, 9.0]])
query = np.array([2.2, 2.4])
nearest = int(np.argmin(np.linalg.norm(X - query, axis=1)))

```

</details>

### Exercise 2 · Medium · Classify with scikit-learn

Fit `KNeighborsClassifier(n_neighbors=3)` and store the predicted class for `new_customer` in `label`. Remember to **scale** first (income is in thousands, age in years) using a pipeline.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
X = np.array([[25, 30000], [30, 32000], [45, 90000], [50, 95000], [28, 31000], [48, 88000]])
y = np.array(["budget", "budget", "premium", "premium", "budget", "premium"])
new_customer = [[46, 85000]]
label = None   # TODO


In [ ]:
try:
    check("premium customer", label == "premium")
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
X = np.array([[25, 30000], [30, 32000], [45, 90000], [50, 95000], [28, 31000], [48, 88000]])
y = np.array(["budget", "budget", "premium", "premium", "budget", "premium"])
new_customer = [[46, 85000]]
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)).fit(X, y)
label = model.predict(new_customer)[0]

```

</details>

### Exercise 3 · Stretch · Choose k with cross-validation

Try every odd `k` from 1 to 15 with 5-fold cross-validated accuracy (scaled features). Store the best `k` in `best_k` and the dict of mean scores in `scores`.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
X, y = make_classification(n_samples=400, n_features=6, n_informative=4, random_state=0)
best_k = None
scores = {}   # TODO


In [ ]:
try:
    check("eight candidates", len(scores) == 8)
    check("best k is odd and in range", best_k in range(1, 16, 2))
    check("good accuracy", max(scores.values()) > 0.8)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
X, y = make_classification(n_samples=400, n_features=6, n_informative=4, random_state=0)
scores = {k: cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), X, y, cv=5).mean() for k in range(1, 16, 2)}
best_k = max(scores, key=scores.get)

```

Small k is flexible and noisy; large k is smooth and biased. Cross-validation finds the compromise.

</details>

---
*Back to the course: **Machine Learning End To End → K-Nearest Neighbours**.*